In [1]:
import torch
from tokenizers import Tokenizer
import numpy as np
from tqdm import tqdm
import math
from collections import defaultdict
from pathlib import Path

In [2]:
TOKENIZER = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/tokenizer.json"

SFT_TRAIN_TOKENS = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/sft_train_tokens.bin"
SFT_TRAIN_MASK = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/sft_train_mask.bin"
SFT_TRAIN_OFFSETS = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/sft_train_offsets.npy"

SFT_VALIDATION_TOKENS = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/sft_valid_tokens.bin"
SFT_VALIDATION_MASK = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/sft_valid_mask.bin"
SFT_VALIDATION_OFFSETS = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/sft_valid_offsets.npy"

SFT_TEST_TOKENS = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/sft_test_tokens.bin"
SFT_TEST_MASK = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/sft_test_mask.bin"
SFT_TEST_OFFSETS = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/sft_test_offsets.npy"

BEST_PRETRAIN_MODEL = '/kaggle/input/datasets/absolute1000/modelpretrain-with-thinking/PRETRAIN_BEST.pt'
torch.manual_seed(0)

In [3]:
tok = Tokenizer.from_file(TOKENIZER)
PAD_ID = 0
MASK_ID = 1
EOS_ID = 4

In [4]:
def RoPE(batch):
    B, L, D = batch.shape
    x1 = batch[..., 0::2]
    x2 = batch[..., 1::2]

    pos = torch.arange(L, device=batch.device)

    freqs = 10000 ** (-torch.arange(0, D, 2, device=batch.device) / D)

    angles = pos[:, None] * freqs[None, :]

    cos = torch.cos(angles).unsqueeze(0)
    sin = torch.sin(angles).unsqueeze(0)

    x1_rot = x1 * cos - x2 * sin
    x2_rot = x1 * sin + x2 * cos

    return torch.stack([x1_rot, x2_rot], dim=-1).flatten(-2)

class SelfAttention(torch.nn.Module):
    def __init__(self, emb_size, head_size):
        super().__init__()
        self.C = math.sqrt(head_size)
        self.Query = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Key = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Value = torch.nn.Linear(emb_size, head_size, bias=False)

    def forward(self, x, mask=None): # x[batch_size, tok_size, emb_size], 
        query, key, value = self.Query(x), self.Key(x), self.Value(x) #[batch, tok_size, head_size]
        query, key = RoPE(query), RoPE(key)
        
        internal_state = query @ key.transpose(-2, -1) #[batch, tok_size, tok_size]
        if mask is not None: internal_state = internal_state.masked_fill(~mask, float('-inf'))
        weights = torch.nn.functional.softmax(internal_state / self.C, dim=-1) #[batch, tok_size, tok_size]
        out = weights @ value
        return out # [batch, tok_size, head_size]

class MultiHeadSelfAttention(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        head_size = emb_size // n_heads
        self.heads = torch.nn.ModuleList([SelfAttention(emb_size, head_size) for head in range(n_heads)])
        self.Output = torch.nn.Linear(emb_size, emb_size, bias=False)

    def forward(self, x, mask=None):
        temporary_states = [head(x, mask) for head in self.heads]
        temporary_states = torch.cat(temporary_states, dim=-1) # [batch, tok_size, head_size] -> [batch, tok_size, emb_size]
        output = self.Output(temporary_states)
        return output # [batch, tok_size, emb_size]

class FeedForward(torch.nn.Module):
    def __init__(self, emb_size, out_size=None):
        super().__init__()
        neurons = int((8 / 3) * emb_size) - (int((8 / 3) * emb_size) % 64)
        if out_size is None: out_size = emb_size
        self.W1 = torch.nn.Linear(emb_size, neurons, bias=False)
        self.SiLU = torch.nn.SiLU()
        self.W2 = torch.nn.Linear(emb_size, neurons, bias=False)
        self.W3 = torch.nn.Linear(neurons, out_size, bias=False)

    def forward(self, x): # [batch, tok_size, emb_size]
        hidden_state = self.SiLU(self.W1(x)) * self.W2(x) # [batch, tok_size, neurons]
        output = self.W3(hidden_state)
        return output # [batch, tok_size, out_size]

class EncoderBlock(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        self.MultiHeadSelfAttention = MultiHeadSelfAttention(emb_size, n_heads)
        self.RMSNorm1 = torch.nn.RMSNorm(emb_size)
        self.FeedForward = FeedForward(emb_size)
        self.RMSNorm2 = torch.nn.RMSNorm(emb_size)

    def forward(self, context, mask): # [batch_size, context_tok_size, emb_size]

        # [batch_size, context_tok_size, emb_size]
        residual_connection1 = self.MultiHeadSelfAttention(self.RMSNorm1(context), mask) + context
        
        # [batch_size, context_tok_size, emb_size]
        residual_connection2 = self.FeedForward(self.RMSNorm2(residual_connection1)) + residual_connection1

        return residual_connection2 # [batch_size, context_tok_size, emb_size]

class Encoder(torch.nn.Module):
    def __init__(self, n_blocks, emb_size, n_heads):
        super().__init__()
        self.blocks = torch.nn.ModuleList([EncoderBlock(emb_size, n_heads) for block in range(n_blocks)])
        self.RMSNorm = torch.nn.RMSNorm(emb_size)

    def forward(self, context, mask):
        internal_state = context
        for encoder in self.blocks:
            internal_state = encoder(internal_state, mask)
        output = self.RMSNorm(internal_state)
        return output # [batch_size, context_tok_size, emb_size]

class Thinking(torch.nn.Module):
    def __init__(self, emb_size, internal_state_size=128, vocab_size=12288, momentum=0.1):
        super().__init__()
        self.momentum = momentum
        self.to_vocab = torch.nn.Linear(emb_size, vocab_size, bias=False)
        self.alpha = torch.nn.Parameter(torch.ones(vocab_size))
        self.SiLU = torch.nn.SiLU()

        self.to_internal = torch.nn.Linear(vocab_size, internal_state_size, bias=False)
        self.ln = torch.nn.LayerNorm(internal_state_size)
        self.to_emb = torch.nn.Linear(internal_state_size, emb_size, bias=False)

        self.register_buffer("e_mean", torch.zeros(vocab_size))
        self.t_in = torch.nn.Linear(vocab_size, internal_state_size, bias=False)
        self.t_ln = torch.nn.LayerNorm(internal_state_size)
        self.t_out = torch.nn.Linear(internal_state_size, emb_size, bias=False)

    def forward(self, x, mask_for_thinking, context_ids):
        # x [B, Cs, Emb], mask_for_thinking [B, Cs, 1], context_ids [B, Cs]
        x_unit = x / x.norm(dim=-1, keepdim=True).clamp(min=1e-6)
        vocab_unit = self.to_vocab.weight / self.to_vocab.weight.norm(dim=-1, keepdim=True).clamp(min=1e-6)
        energy = x_unit @ vocab_unit.T # [B, Cs, V], from -1 to 1

        own = torch.zeros_like(energy, dtype=torch.bool).scatter_(2, context_ids[..., None], True)
        drop = (~mask_for_thinking) | own # padding + the position's own token
        kept = energy.masked_fill(drop, 0.0) # [B, Cs, V]

        # words channel
        internal_state = kept * self.alpha
        internal_state = self.to_internal(internal_state)
        internal_state = self.ln(internal_state)
        internal_state = self.SiLU(internal_state)
        words = self.to_emb(internal_state) # [B, Cs, Emb]

        # thought channel
        e = kept.sum(dim=1).float() # [B, V]: sum of energies over positions
        if self.training:
            mu = e.mean(dim=0) # mean over texts in the batch
            with torch.no_grad():
                self.e_mean.mul_(1 - self.momentum).add_(self.momentum * mu)
        else:
            mu = self.e_mean # inference: running mean
        e = e - mu # how this text differs from the others
        e = e / e.pow(2).mean(dim=-1, keepdim=True).sqrt().clamp(min=1e-6) # unit scale

        thought = self.t_in(e)
        thought = self.t_ln(thought)
        thought = self.SiLU(thought)
        thought = self.t_out(thought) # [B, Emb]

        thinking_out = words + thought.unsqueeze(1) # thought is added to every position
        return thinking_out

class CrossAttention(torch.nn.Module):
    def __init__(self, emb_size, head_size):
        super().__init__()
        self.C = math.sqrt(head_size)
        self.Query = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Key = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Value = torch.nn.Linear(emb_size, head_size, bias=False)

    def forward(self, data_for_query, data_for_key_value, mask=None):
        # data_for_query[batch, answer_tok_size, emb_size]
        # data_for_key_value[batch, context_tok_size, emb_size] 
        
        query, key, value = self.Query(data_for_query), self.Key(data_for_key_value), self.Value(data_for_key_value)
        # query[batch, answer_tok_size, head_size]
        # key, value[batch, context_tok_size, head_size]
        
        internal_state = query @ key.transpose(-2, -1) #[batch, answer_tok_size, context_tok_size]
        if mask is not None: internal_state = internal_state.masked_fill(~mask, float('-inf'))
        weights = torch.nn.functional.softmax(internal_state / self.C, dim=-1)
        out = weights @ value
        return out #[batch, answer_tok_size, head_size]

class MultiHeadCrossAttention(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        head_size = emb_size // n_heads
        self.heads = torch.nn.ModuleList([CrossAttention(emb_size, head_size) for head in range(n_heads)])
        self.Output = torch.nn.Linear(emb_size, emb_size, bias=False)

    def forward(self, data_for_query, data_for_key_value, mask=None):
        temporary_states = [head(data_for_query, data_for_key_value, mask) for head in self.heads]
        temporary_states = torch.cat(temporary_states, dim=-1) # [batch, answer_tok_size, head_size] -> [batch, answer_tok_size, emb_size]
        output = self.Output(temporary_states)
        return output # [batch, answer_tok_size, emb_size]

class DecoderBlock(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        self.MultiHeadSelfAttention = MultiHeadSelfAttention(emb_size, n_heads)
        self.RMSNorm1 = torch.nn.RMSNorm(emb_size)
        self.MultiHeadCrossAttention = MultiHeadCrossAttention(emb_size, n_heads)
        self.RMSNorm2 = torch.nn.RMSNorm(emb_size)
        self.RMSNorm3 = torch.nn.RMSNorm(emb_size)
        self.FeedForward = FeedForward(emb_size)

    def forward(self, encoder_output, answer, encoder_mask=None):
        
        # [batch, answer_tok_size, emb_size]
        internal_state = self.MultiHeadSelfAttention(self.RMSNorm1(answer))
        residual_connection1 =  internal_state + answer
        internal_state = self.MultiHeadCrossAttention(
            self.RMSNorm2(residual_connection1), encoder_output, encoder_mask) # [batch, answer_tok_size, emb_size]
        residual_connection2 = internal_state + residual_connection1
        internal_state = self.FeedForward(self.RMSNorm3(residual_connection2))
        residual_connection3 = internal_state + residual_connection2 # [batch, answer_tok_size, emb_size]
        return residual_connection3 # [batch, answer_tok_size, emb_size]
        
class Decoder(torch.nn.Module):
    def __init__(self, n_blocks, emb_size, n_heads):
        super().__init__()
        self.blocks = torch.nn.ModuleList([DecoderBlock(emb_size, n_heads) for block in range(n_blocks)])
        self.RMSNorm = torch.nn.RMSNorm(emb_size)
        self.final = torch.nn.Linear(emb_size, 12288, bias=False)

    def forward(self, answer, encoder_output, mask):
        internal_state = answer
        for decoder in self.blocks:
            internal_state = decoder(encoder_output, internal_state, mask) # [batch_size, answer_size, emb_size]
        internal_state = self.RMSNorm(internal_state)
        internal_state = self.final(internal_state)
        return internal_state # [batch_size, answer_size, dict_size]

class NN(torch.nn.Module):
    def _init_weights(self, m):
        if isinstance(m, (torch.nn.Linear, torch.nn.Embedding)):
            torch.nn.init.normal_(m.weight, mean=0.0, std=0.02)
        
    def __init__(self, 
                 embedding_size, 
                 encoder_num, 
                 heads_encoder,
                 thinking_internal_state_size, 
                 decoder_num, 
                 heads_decoder):
        super().__init__()
        
        self.Embedding = torch.nn.Embedding(12288, embedding_size)
        self.Encoder = Encoder(encoder_num, embedding_size, heads_encoder)
        self.Thinking = Thinking(embedding_size, thinking_internal_state_size)
        self.Thinking.to_vocab.weight = self.Embedding.weight
        self.out_norm = torch.nn.RMSNorm(embedding_size)
        self.Decoder = Decoder(decoder_num, embedding_size, heads_decoder)
        self.Decoder.final.weight = self.Embedding.weight
        self.apply(self._init_weights)
        torch.nn.init.zeros_(self.Thinking.t_out.weight)

    def encode(self, context_ids):
        # context_ids [B, Cs]
        PAD_ID = 0
        mask = (context_ids != PAD_ID)[:, None, :] # [B, 1, Cs]
        memory = self.Encoder(self.Embedding(context_ids), mask) # [B, Cs, Emb]
        memory = self.Thinking(memory, mask.transpose(1, 2), context_ids)
        memory = self.out_norm(memory)
        return memory, mask
    
    def decode(self, answer_ids, memory, mask):
        return self.Decoder(self.Embedding(answer_ids), memory, mask) # [B, La, V]
    
    def forward(self, context_ids, answer_ids):
        memory, mask = self.encode(context_ids)
        return self.decode(answer_ids, memory, mask)

In [5]:
class SFTDataset(torch.utils.data.Dataset):

    def __init__(self, tokens_path, mask_path, offsets_path):
        self.tokens_path = tokens_path
        self.mask_path = mask_path
        self.offsets_path = offsets_path
        self.tokens = np.memmap(self.tokens_path, dtype=np.uint16, mode="r")
        self.mask = np.memmap(self.mask_path, dtype=np.uint8, mode="r")
        self.offsets = np.load(self.offsets_path)
        self.length = len(self.offsets) - 1

    def __getitem__(self, index):
        li = self.offsets[index]
        ri = self.offsets[index + 1]

        dialogue = self.tokens[li:ri]
        mask = self.mask[li:ri]

        temp_helper = np.diff(mask)
        bot_phrases_starts_idx = np.where(temp_helper==1)[0]
        bot_phrases_starts_idx += np.ones(len(bot_phrases_starts_idx), dtype=np.int64)
        bot_phrases_ends_idx = np.where(temp_helper==255)[0]
        bot_phrases_ends_idx += np.ones(len(bot_phrases_ends_idx), dtype=np.int64)
        
        random_start = bot_phrases_starts_idx[torch.randint(len(bot_phrases_starts_idx), (1,)).item()]
        end_of_chosen_bot_phrase_idx = np.where(bot_phrases_ends_idx>random_start)[0]
        
        context = dialogue[:random_start]
        context_mask = mask[:random_start]
        if end_of_chosen_bot_phrase_idx.size == 0:
            bot_tokens = dialogue[random_start:]
            bot_mask = mask[random_start:]
        else:
            idx = end_of_chosen_bot_phrase_idx[0]
            bot_tokens = dialogue[random_start:bot_phrases_ends_idx[idx]]
            bot_mask = mask[random_start:bot_phrases_ends_idx[idx]]

        context = torch.from_numpy(context.astype(np.int64))
        bot_tokens = torch.from_numpy(bot_tokens.astype(np.int64))
        return context, bot_tokens
    def __len__(self):
        return self.length

In [6]:
def collate_fn(list_of_samples):
    #tok.encode("<pad>").ids = [0]
    
    B = len(list_of_samples)
    _batch_samples = torch.zeros((B, 1000), dtype=torch.int64)
    _batch_targets = torch.zeros((B, 500), dtype=torch.int64)
    for idx, data in enumerate(list_of_samples):
        context, bot_tokens = data
        _batch_samples[idx, -len(context):] = context
        _batch_targets[idx, :len(bot_tokens)] = bot_tokens
    return _batch_samples, _batch_targets

In [7]:
MODEL_SETTINGS_CONFIG = {
    "embedding_size": 384,
    "encoder_num": 4,
    "heads_encoder": 6,
    "thinking_internal_state_size": 128,
    "decoder_num": 2,
    "heads_decoder": 6,
}

In [8]:
def make_record(
    reason, step, model, optimizer, scheduler, scaler, best_loss,
    context_size, answer_size, train_loss_history, valid_loss_history, MODEL_SETTINGS_CONFIG
):
    path = f'/kaggle/working/SFT_{reason}.pt'
    Path(path).unlink(missing_ok=True)
    torch.save({
        'step': step,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler': scaler.state_dict(),
        'best_loss': best_loss,
        'context_size': context_size,
        'answer_size': answer_size,
        'train_loss_history': dict(train_loss_history),
        'valid_loss_history': dict(valid_loss_history),
        'MODEL_SETTINGS_CONFIG': MODEL_SETTINGS_CONFIG
    }, path)

In [9]:
def add_noise(tokens, t):
    #tok.encode("<mask>").ids = 1
    #tokens[samples, window_size]
    #t[samples]
    batch_len, tokens_per_sample = tokens.shape
    noise = torch.rand(batch_len, tokens_per_sample, device=tokens.device) #noise[samples, answer_window]
    noise = noise < t.unsqueeze(-1)
    masked = torch.where(noise, 1, tokens)
    return masked, noise

In [10]:
def Loss(predictions, ans, noise, answer_window, t):
    #predictions[samples, 12288, answer_window], ans[samples, answer_window], noise[samples, answer_window]
    #answer_window[1], t[samples]
    loss = torch.nn.functional.cross_entropy(predictions, ans, reduction='none') #loss[samples, answer_window]
    loss = torch.where(noise, loss, 0) #loss[samples, answer_window]
    loss = torch.sum(loss, dim=1) #loss[samples]
    loss /= t
    loss /= answer_window
    return torch.mean(loss)

In [11]:
PROMTS = [
    'Hello there! How are you?',
    'Do you think about Roman Empire daily?',
    'Weather is so nice these seasons.',
    'I think an AI or Math student can easily turn himself into quant or hft sphere.',
    "Let's build an AI-native SaaS startup toghether."
]
USER_ID, BOT_ID = tok.token_to_id("<user>"), tok.token_to_id("<bot>")

@torch.no_grad()
def generate_reply(model, text, device, answer_len=64, temperature=0.7, top_k=50, block_size=8, rep_penalty=1.3):
    ids = [USER_ID] + tok.encode(text).ids + [BOT_ID]
    ctx = torch.tensor([ids], device=device)
    KV_CACHE, KV_mask = model.encode(ctx)
    ans = torch.full((1, answer_len), MASK_ID, dtype=torch.long, device=device)
    pos = torch.arange(answer_len, device=device)
    for step in range(answer_len * 2):
        is_mask = ans[0] == MASK_ID
        n_left = int(is_mask.sum())
        if n_left == 0:
            break
        logits = model.decode(ans, KV_CACHE, KV_mask)[0].float()
        logits[:, MASK_ID] = -float("inf")
        used = ans[0][~is_mask].unique()
        if rep_penalty > 1.0 and len(used):
            lu = logits[:, used]
            logits[:, used] = torch.where(lu > 0, lu / rep_penalty, lu * rep_penalty)
        logits = logits / temperature
        kth = logits.topk(top_k, dim=-1).values[:, -1:]
        probs = torch.softmax(logits.masked_fill(logits < kth, -float("inf")), -1)
        pred = torch.multinomial(probs, 1).squeeze(-1)
        conf = probs.gather(-1, pred[:, None]).squeeze(-1)
        first_still_masked = int(is_mask.nonzero()[0])
        eligible = is_mask & (pos < (first_still_masked // block_size + 1) * block_size)
        conf = conf.masked_fill(~eligible, -1.0)
        k = min(max(1, math.ceil(n_left / max(1, answer_len - step))), int(eligible.sum()))
        idx = conf.topk(k).indices
        ans[0, idx] = pred[idx]
    out = ans[0].tolist()
    if EOS_ID in out:
        out = out[:out.index(EOS_ID)]
    return tok.decode(out)

def generation_test(model, device, answer_len):
    for p in PROMTS:
        torch.manual_seed(0)
        print(f"USER: {p}\nBOT:  {generate_reply(model, p, device, answer_len)}\n")

In [12]:
def TRAIN_PIPELINE_SFT_MODE(
    EPOCH_NUM=None,
    START_EPOCH=0,
    BATCH_SIZE=64,
    context_size = None,
    answer_size = None,
    lr = 1e-4,
    weight_decay = 0.1,
    warmup_steps = 100,
    best_loss = float('inf'),
    init_path = None,
    resume_path = None
):
    TRAIN_DATALOADER = torch.utils.data.DataLoader(
        SFTDataset(SFT_TRAIN_TOKENS, SFT_TRAIN_MASK, SFT_TRAIN_OFFSETS), BATCH_SIZE, shuffle=True, num_workers=4,
        collate_fn=collate_fn, pin_memory=True, drop_last=True, persistent_workers=True
    )
    VALIDATION_DATALOADER = torch.utils.data.DataLoader(
        SFTDataset(SFT_VALIDATION_TOKENS, SFT_VALIDATION_MASK, SFT_VALIDATION_OFFSETS), BATCH_SIZE, num_workers=4,
        collate_fn=collate_fn, pin_memory=True, drop_last=True, persistent_workers=True
    )

    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    ckpt = torch.load(resume_path or init_path, map_location=DEVICE, weights_only=False) if (resume_path or init_path) else None
    config = ckpt['MODEL_SETTINGS_CONFIG'] if ckpt else MODEL_SETTINGS_CONFIG
    model = NN(**config).to(DEVICE)

    decay    = [p for p in model.parameters() if p.dim() >= 2]
    no_decay = [p for p in model.parameters() if p.dim() < 2]
    optimizer = torch.optim.AdamW(
    [{"params": decay, "weight_decay": weight_decay},
     {"params": no_decay, "weight_decay": 0.0}], lr=lr)
    warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.01, total_iters=warmup_steps)
    cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCH_NUM * len(TRAIN_DATALOADER) - warmup_steps)
    scaler = torch.amp.GradScaler('cuda')
    scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_steps])

    if ckpt:
        model.load_state_dict(ckpt['model'])
    if resume_path:
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        scaler.load_state_dict(ckpt['scaler'])
        START_EPOCH = ckpt['step'] + 1
        best_loss = ckpt['best_loss']
        context_size = ckpt['context_size']
        answer_size = ckpt['answer_size']

    train_loss_history = defaultdict(list, ckpt['train_loss_history']) if resume_path else defaultdict(list)
    valid_loss_history = defaultdict(list, ckpt['valid_loss_history']) if resume_path else defaultdict(list)

    for epoch in range(START_EPOCH, EPOCH_NUM):
        TRAIN_CYCLE = tqdm(TRAIN_DATALOADER, desc=f"TRAIN {epoch}")
        for batch in TRAIN_CYCLE:
            model.train()
            batch_context, batch_target = batch[0][:, -context_size:].to(DEVICE), batch[1][:, :answer_size].to(DEVICE)
            batch_noise_schedule = ((torch.arange(BATCH_SIZE, device=DEVICE) + torch.rand(1, device=DEVICE)) / BATCH_SIZE).clamp(min=1e-3)
            masked_answer_input, noise_mask = add_noise(batch_target, batch_noise_schedule)
            
            with torch.amp.autocast('cuda'):
                out = model(batch_context, masked_answer_input).transpose(1, 2)
                loss = Loss(out, batch_target, noise_mask, answer_size, batch_noise_schedule)
            
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            TRAIN_CYCLE.set_postfix({'loss': f'{loss.item():.4f}', 'lr': f'{scheduler.get_last_lr()[0]:.2e}'})
            train_loss_history[epoch].append(loss.item())

        VALIDATION_CYCLE = tqdm(VALIDATION_DATALOADER, desc="VALID")
        model.eval()
        with torch.no_grad():
            for batch in VALIDATION_CYCLE:
                batch_context, batch_target = batch[0][:, -context_size:].to(DEVICE), batch[1][:, :answer_size].to(DEVICE)
                batch_noise_schedule = ((torch.arange(BATCH_SIZE, device=DEVICE) + torch.rand(1, device=DEVICE)) / BATCH_SIZE).clamp(min=1e-3)
                masked_answer_input, noise_mask = add_noise(batch_target, batch_noise_schedule)
                
                with torch.amp.autocast('cuda'):
                    out = model(batch_context, masked_answer_input).transpose(1, 2)
                    loss = Loss(out, batch_target, noise_mask, answer_size, batch_noise_schedule)
                valid_loss_history[epoch].append(loss.item())

            make_record("RECORD", epoch, model, optimizer, scheduler, scaler, best_loss, context_size, answer_size, 
                        train_loss_history, valid_loss_history, config)
            
            mean_val_loss_per_epoch = torch.tensor(valid_loss_history[epoch]).mean()
            print(f"epoch {epoch}: val {mean_val_loss_per_epoch:.4f}")
            if mean_val_loss_per_epoch < best_loss:
                best_loss = mean_val_loss_per_epoch.item()
                make_record("BEST", epoch, model, optimizer, scheduler, scaler, best_loss, context_size, answer_size, 
                        train_loss_history, valid_loss_history, config)
            generation_test(model, DEVICE, answer_size)
    return model

In [13]:
model = TRAIN_PIPELINE_SFT_MODE(
    EPOCH_NUM=20,
    START_EPOCH=0,
    BATCH_SIZE=96,
    context_size = 256,
    answer_size = 64,
    lr = 1e-4,
    weight_decay = 0.1,
    warmup_steps = 100,
    best_loss = float('inf'),
    init_path = BEST_PRETRAIN_MODEL,
    resume_path = None
)

TRAIN 0:   0%|          | 0/455 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:2954: UserWarning: Mismatch dtype between input and weight: input dtype = c10::Half, weight dtype = float, Cannot dispatch to fused implementation. (Triggered internally at /pytorch/aten/src/ATen/native/layer_norm.cpp:344.)
  return torch.rms_norm(input, normalized_shape, weight, eps)
/usr/local/lib/python3.13/dist-packages/torch/optim/lr_scheduler.py:1195: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
VALID: 100%|██████████| 7/7 [00:01<00:00,  4.95it/s]


epoch 0: val 0.9587
USER: Hello there! How are you?
BOT:  

USER: Do you think about Roman Empire daily?
BOT:  

USER: Weather is so nice these seasons.
BOT:  I am so sorry.

USER: I think an AI or Math student can easily turn himself into quant or hft sphere.
BOT:  

USER: Let's build an AI-native SaaS startup toghether.
BOT:  Yes.



VALID: 100%|██████████| 7/7 [00:01<00:00,  5.31it/s]


epoch 1: val 0.9364
USER: Hello there! How are you?
BOT:  .

USER: Do you think about Roman Empire daily?
BOT:  My favorite.

USER: Weather is so nice these seasons.
BOT:  

USER: I think an AI or Math student can easily turn himself into quant or hft sphere.
BOT:  

USER: Let's build an AI-native SaaS startup toghether.
BOT:  Cool.



VALID: 100%|██████████| 7/7 [00:01<00:00,  5.34it/s]


...

epoch 17: val 0.9351
USER: Hello there! How are you?
BOT:  

USER: Do you think about Roman Empire daily?
BOT:  

USER: Weather is so nice these seasons.
BOT:  

USER: I think an AI or Math student can easily turn himself into quant or hft sphere.
BOT:  What's that?

USER: Let's build an AI-native SaaS startup toghether.
BOT:  



VALID: 100%|██████████| 7/7 [00:01<00:00,  5.33it/s]


epoch 18: val 0.9369
USER: Hello there! How are you?
BOT:  

USER: Do you think about Roman Empire daily?
BOT:  Of course.

USER: Weather is so nice these seasons.
BOT:  

USER: I think an AI or Math student can easily turn himself into quant or hft sphere.
BOT:  `

USER: Let's build an AI-native SaaS startup toghether.
BOT:  



VALID: 100%|██████████| 7/7 [00:01<00:00,  5.31it/s]


epoch 19: val 0.9482
USER: Hello there! How are you?
BOT:  Hi how are you?

USER: Do you think about Roman Empire daily?
BOT:  

USER: Weather is so nice these seasons.
BOT:  

USER: I think an AI or Math student can easily turn himself into quant or hft sphere.
BOT:  What's the math?

USER: Let's build an AI-native SaaS startup toghether.
BOT:  



In [14]:
TEST_DATALOADER = torch.utils.data.DataLoader(
        SFTDataset(SFT_TEST_TOKENS, SFT_TEST_MASK, SFT_TEST_OFFSETS), 96, shuffle=True, num_workers=4,
        collate_fn=collate_fn, pin_memory=True, drop_last=False, persistent_workers=True
    )
TEST_CYCLE = tqdm(TEST_DATALOADER, desc="TEST")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
config = torch.load('/kaggle/working/SFT_BEST.pt', map_location=DEVICE, weights_only=False)
model = NN(**config['MODEL_SETTINGS_CONFIG']).to(DEVICE)
model.load_state_dict(config['model'])
model.eval()
bn = 0
test_losses = []
with torch.no_grad():
    for batch in TEST_CYCLE:
        batch_context, batch_target = batch[0][:, -256:].to(DEVICE), batch[1][:, :64].to(DEVICE)
        b = batch_target.shape[0]
        batch_noise_schedule = ((torch.arange(b, device=DEVICE) + torch.rand(1, device=DEVICE)) / b).clamp(min=1e-3)
        masked_answer_input, noise_mask = add_noise(batch_target, batch_noise_schedule)
        
        with torch.amp.autocast('cuda'):
            out = model(batch_context, masked_answer_input).transpose(1, 2)
            loss = Loss(out, batch_target, noise_mask, 64, batch_noise_schedule)

        TEST_CYCLE.set_postfix({'batch': f'{bn}', 'loss': f'{loss.item():.4f}'})
        test_losses.append(loss.item())
        bn += 1
print(f"TEST loss: {np.mean(test_losses):.4f}")

TEST: 100%|██████████| 8/8 [00:02<00:00,  3.77it/s, batch=7, loss=1.4978]

TEST loss: 1.0338
